<a href="https://colab.research.google.com/github/ashu9439/AI_Lab/blob/main/gen_ai___RAG___step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
gorq_cloud_ApiKey = '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'

In [44]:
# ChatGroq from langchain_groq, which allows you to use Groq-hosted LLMs in LangChain.

!pip install langchain_groq

from langchain_groq import ChatGroq


In [46]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key= gorq_cloud_ApiKey,
    verbose=True,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


response = llm.invoke('explain me Gen Ai in 2 lines')
print(response.content)

Gen AI, or General Artificial Intelligence, refers to a type of AI that can perform any intellectual task that a human can, with the ability to learn, reason, and apply knowledge across a wide range of domains. It aims to create a machine that can think, learn, and behave like a human being, with capabilities that surpass those of narrow or specialized AI systems.


# Create a Vector DB

In [47]:
!pip install chromadb

import chromadb

# Initialize Chroma client.
client = chromadb.Client()


In [48]:
# Create a new collection.
collection = client.create_collection("my_collection")

UniqueConstraintError: Collection my_collection already exists

In [49]:

# Add some documents to the collection.
collection.add(
    documents=[
        "This is the first document.",
        "This is the second document.",
        "My name is Ashutosh.",
        "I play cricket.",
        "I love samosa."
    ],
    ids=["id1", "id2", "id3", "id4", "id5"],
    metadatas=[
        {"source": "file1"},
        {"source": "file2"},
        {"source": "user_input"},
        {"source": "user_input"},
        {"source": "user_input"}
    ],
)


In [50]:
# Query the collection
results = collection.query(
    query_texts=["bada pav."],
    n_results=2
)

results

{'ids': [['id5', 'id3']],
 'embeddings': None,
 'documents': [['I love samosa.', 'My name is Ashutosh.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'source': 'user_input'}, {'source': 'user_input'}]],
 'distances': [[1.5492101470875044, 1.7423234016510527]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [51]:
'''
Imports WebBaseLoader → This is a document loader from LangChain that fetches web page content.
Loads the webpage (https://jobs.nike.com/job/R-33460).
Extracts the page content and stores it in page_data.
Prints the extracted content.
'''
!pip install langchain_community
from langchain_community.document_loaders import WebBaseLoader




In [52]:
loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
page_data = loader.load().pop().page_content
print(page_data)























Nike Careers








































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Language

  Deutsc

In [53]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)
print(res.content)

```json
[
  {
    "role": "ATHLETE III",
    "experience": null,
    "skills": null,
    "description": "Job Location: Nanjing, Jiangsu, China, Retail Stores"
  },
  {
    "role": "Abteilungsleiter/in (Coach) - 100% - NFS Landquart",
    "experience": null,
    "skills": null,
    "description": "Remote Job Location: Landquart, Graubunden, Switzerland, Retail Stores"
  },
  {
    "role": "Account Partner Representative, Running",
    "experience": null,
    "skills": null,
    "description": "Job Location: Toronto, Ontario, Canada, Sales"
  },
  {
    "role": "Addetto vendite/Athlete - Part Time 20 h NUS CASTEL ROMANO",
    "experience": null,
    "skills": null,
    "description": "Job Location: Castel Romano, Roma, Italy, Retail Stores"
  },
  {
    "role": "Addetto vendite/Athlete - Part Time 20h NFS MILANO SCALO",
    "experience": null,
    "skills": null,
    "description": "Job Location: Milano, Milano, Italy, Retail Stores"
  },
  {
    "role": "Allocator, NA",
    "experience"

In [54]:
# convert string to JSON

from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

type(json_res)

list

In [55]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file (update the path accordingly)
file_path = "/content/drive/My Drive/my_portfolio.csv"

# Load the CSV file into a Pandas DataFrame
df = pd.read_csv(file_path)

# Display the DataFrame
df.head()  # Show the first few rows


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio


In [56]:
# add the port folios to the vector db so that we can retrive later

import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [57]:
links = collection.query(query_texts=['experience in python', 'javascript'],
                        n_results=2)
links

{'ids': [['80eced44-aa63-46b8-9cab-d09613593d90',
   '264a741b-2185-43ee-86f5-d9ed12d2a0d6'],
  ['b8a4fa3a-8dec-42ae-9374-dbce83762924',
   '03ea27f8-4856-403e-9ea5-051c272d8729']],
 'embeddings': None,
 'documents': [['Machine Learning, Python, TensorFlow',
   'Python, Django, MySQL'],
  ['Full-stack, JavaScript, Express.js', 'Frontend, TypeScript, Angular']],
 'uris': None,
 'data': None,
 'metadatas': [[{'links': 'https://example.com/ml-python-portfolio'},
   {'links': 'https://example.com/python-portfolio'}],
  [{'links': 'https://example.com/full-stack-js-portfolio'},
   {'links': 'https://example.com/typescript-frontend-portfolio'}]],
 'distances': [[0.9968848007555503, 1.0578701721493369],
  [1.0805880816108433, 1.5001293111238974]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [58]:
links.get('metadatas', [])

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}]]

In [60]:
job = json_res
job


[{'role': 'ATHLETE III',
  'experience': None,
  'skills': None,
  'description': 'Job Location: Nanjing, Jiangsu, China, Retail Stores'},
 {'role': 'Abteilungsleiter/in (Coach) - 100% - NFS Landquart',
  'experience': None,
  'skills': None,
  'description': 'Remote Job Location: Landquart, Graubunden, Switzerland, Retail Stores'},
 {'role': 'Account Partner Representative, Running',
  'experience': None,
  'skills': None,
  'description': 'Job Location: Toronto, Ontario, Canada, Sales'},
 {'role': 'Addetto vendite/Athlete - Part Time 20 h NUS CASTEL ROMANO',
  'experience': None,
  'skills': None,
  'description': 'Job Location: Castel Romano, Roma, Italy, Retail Stores'},
 {'role': 'Addetto vendite/Athlete - Part Time 20h NFS MILANO SCALO',
  'experience': None,
  'skills': None,
  'description': 'Job Location: Milano, Milano, Italy, Retail Stores'},
 {'role': 'Allocator, NA',
  'experience': None,
  'skills': None,
  'description': 'Job Location: Beaverton, Oregon, United States, S

In [61]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhance Your Business Operations with AtliQ's Automated Solutions

Dear Hiring Manager,

I came across the various job openings at Nike, including ATHLETE III, Abteilungsleiter/in (Coach), Account Partner Representative, and more, and I was impressed by the company's commitment to innovation and excellence. As a Business Development Executive at AtliQ, I believe our AI and software consulting services can help Nike streamline its business processes, optimize operations, and reduce costs.

At AtliQ, we have a proven track record of empowering enterprises with tailored solutions that foster scalability, process optimization, and heightened overall efficiency. Our expertise in Machine Learning, Python, and TensorFlow can help Nike develop predictive models to forecast sales, optimize inventory management, and improve customer experiences. Additionally, our experience in Full-stack development, JavaScript, and Express.js can enhance Nike's e-commerce platform, providing a seamless